In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from infotaxis import one_target, hex_ops

In [2]:
# Path to save figs and data for figs
path_data = Path("../simulations")

## Set params

In [3]:
cr_all = list(range(5, 11))
br_all = [1, 2]
pm = 0.001

pfa_full = np.hstack((
    np.arange(0, 0.001, 0.0001),
    np.arange(0.001, 0.021, 0.001)
))

In [4]:
path_csv = path_data / f"fig_7BCD_h_est_pm{pm:.0e}"
if not path_csv.exists():
    path_csv.mkdir(parents=True, exist_ok=True)

In [5]:
def get_cross_over_idx(
    pfa_array, cr, br, pm, center_cube=(0, 0, 0), neighbor_cube = (1, 0, -1)
):

    # Compute h_est for repeating or moving beam aim (h_est_c, h_est_n)
    h_est_c_all = []
    h_est_n_all = []

    for pfa in pfa_array:
        print("--------------------------------")
        print(f"pfa={pfa}")

        search_dict = None
        param_echo = {
            str(br): {
                "pm_const": pm,
                "pfa_const": pfa,
            },
        }

        oth = one_target.OneTargetHex(
            search_rule="infotaxis",
            canvas_radius=cr,
            beam_radius=[br],
            target_cube=(-2, 3, -1),
            aim_start_cube=center_cube, # assign first beam aim
            param_animal= param_echo,
            param_echo= param_echo,
            search_dict=search_dict
        )

        # Assign echo outcome
        oth.echo_value = True # receive echo
        oth.echo_type = True  # correct return

        # Update map after receiving echo
        oth.update_X1()
        oth.get_est_ph()

        # Get sequence index of the center and neighbor grids
        k_center = hex_ops.axial_to_k(hex_ops.cube_to_axial(center_cube), oth.canvas_axial)
        k_neighbor = hex_ops.axial_to_k(hex_ops.cube_to_axial(neighbor_cube), oth.canvas_axial)

        # Find flat index of where k_center and k_neighbor in oth.h_est[br]
        seq_idx_center = np.where(oth.k_canvas==k_center)[0][0]
        seq_idx_neighbor = np.where(oth.k_canvas==k_neighbor)[0][0]

        # Get expected entropy for aiming at the center and neighbor grids
        h_est_c = oth.h_est[br][seq_idx_center]
        h_est_n = oth.h_est[br][seq_idx_neighbor]

        # Store results
        h_est_c_all.append(h_est_c)
        h_est_n_all.append(h_est_n)

    h_est_c_all = np.array(h_est_c_all)
    h_est_n_all = np.array(h_est_n_all)

    # Get cross over start point
    # -1 because cross over happens AFTER h_est_n_all > h_est_c_all
    idx_fine_start = (h_est_n_all < h_est_c_all).sum() -1

    return idx_fine_start, h_est_c_all, h_est_n_all

In [6]:
for cr in cr_all:
    for br in br_all:

        print("=========================================================================")
        print(f"cr={cr}, br={br}")
        idx_x, h_est_c, h_est_n = get_cross_over_idx(pfa_array=pfa_full, cr=cr, br=br, pm=pm)

        # Save repeat the same grid (h_est_c) and move to neighbor (h_est_n)
        df = pd.DataFrame(
            [pfa_full, h_est_c, h_est_n],
            index=["pfa", "h_est_c", "h_est_n"]
        ).T
        df["cr"] = cr
        df["br"] = br
        fname = f"cr{cr}_br{br}_pfa_full_range.csv"
        df.to_csv(path_csv / fname)

cr=5, br=1
--------------------------------
pfa=0.0
Initial beam aim is given, but not initial beam radius.
Set initial beam radius to the largest of beam radius choices: 1
--------------------------------
pfa=0.0001
Initial beam aim is given, but not initial beam radius.
Set initial beam radius to the largest of beam radius choices: 1
--------------------------------
pfa=0.0002
Initial beam aim is given, but not initial beam radius.
Set initial beam radius to the largest of beam radius choices: 1
--------------------------------
pfa=0.00030000000000000003
Initial beam aim is given, but not initial beam radius.
Set initial beam radius to the largest of beam radius choices: 1
--------------------------------
pfa=0.0004
Initial beam aim is given, but not initial beam radius.
Set initial beam radius to the largest of beam radius choices: 1
--------------------------------
pfa=0.0005
Initial beam aim is given, but not initial beam radius.
Set initial beam radius to the largest of beam radi